# BTCUSD RSI Mean Reversion Scalp Backtest

This notebook runs a simple long-and-short backtest on BTCUSD using 5-minute bars over the last 7 days.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from backtesting_tools import build_backtest_bundle

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


In [ ]:
backtest_config = {
    "symbol": "BTCUSD",
    "timeframe": "5Min",
    "lookback_days": 7,
    "qty": 0.001,
    "cash": 10000,
    "commission": 0.0,
    "trade_on_close": True,
    "exclusive_orders": True,
    "strategy_params": {
        "rsi_window": 5,
        "oversold_threshold": 30,
        "overbought_threshold": 70,
        "exit_threshold": 50,
        "short_exit_threshold": 50,
        "stop_loss_pct": 0.004,
        "take_profit_pct": 0.006,
    },
}

pd.DataFrame([backtest_config])


In [ ]:
bundle = build_backtest_bundle(config=backtest_config)

settings_df = bundle["settings_df"]
metrics_df = bundle["metrics_df"]
trades_df = bundle["trades_df"]
equity_curve_df = bundle["equity_curve_df"]
signals_df = bundle["signals_df"]
price_df = bundle["price_df"]


In [ ]:
metrics_df


In [ ]:
trades_df.tail(20)


In [ ]:
equity_curve_df.tail(20)


In [ ]:
signals_df.tail(20)


In [ ]:
price_df.tail(20)


In [ ]:
if not equity_curve_df.empty and {"timestamp", "Equity"}.issubset(equity_curve_df.columns):
    equity_plot_df = equity_curve_df.copy()
    equity_plot_df["timestamp"] = pd.to_datetime(equity_plot_df["timestamp"], errors="coerce")
    equity_plot_df = equity_plot_df.dropna(subset=["timestamp", "Equity"])

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(equity_plot_df["timestamp"], equity_plot_df["Equity"], color="navy", linewidth=1.5)
    ax.set_title("Equity Curve")
    ax.set_xlabel("Time")
    ax.set_ylabel("Equity ($)")
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
else:
    print("No equity curve data available to plot.")


In [ ]:
if not price_df.empty and {"timestamp", "close"}.issubset(price_df.columns):
    price_plot_df = price_df.copy()
    price_plot_df["timestamp"] = pd.to_datetime(price_plot_df["timestamp"], utc=True, errors="coerce")
    price_plot_df["close"] = pd.to_numeric(price_plot_df["close"], errors="coerce")
    price_plot_df = price_plot_df.dropna(subset=["timestamp", "close"]).sort_values("timestamp")

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(price_plot_df["timestamp"], price_plot_df["close"], color="black", linewidth=1.0, label="Close")

    if not trades_df.empty:
        plot_trades_df = trades_df.copy()
        for column in ["EntryTime", "ExitTime"]:
            if column in plot_trades_df.columns:
                plot_trades_df[column] = pd.to_datetime(plot_trades_df[column], utc=True, errors="coerce")

        long_trades_df = plot_trades_df[plot_trades_df["Size"] > 0]
        short_trades_df = plot_trades_df[plot_trades_df["Size"] < 0]

        if not long_trades_df.empty:
            ax.scatter(long_trades_df["EntryTime"], long_trades_df["EntryPrice"], color="green", marker="^", s=50, label="Long Entry")
            ax.scatter(long_trades_df["ExitTime"], long_trades_df["ExitPrice"], color="darkgreen", marker="v", s=50, label="Long Exit")

        if not short_trades_df.empty:
            ax.scatter(short_trades_df["EntryTime"], short_trades_df["EntryPrice"], color="red", marker="v", s=50, label="Short Entry")
            ax.scatter(short_trades_df["ExitTime"], short_trades_df["ExitPrice"], color="darkred", marker="^", s=50, label="Short Exit")

    ax.set_title("BTCUSD Price With Backtest Trades")
    ax.set_xlabel("Time")
    ax.set_ylabel("Price")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
else:
    print("No price data available to plot.")
